# Student Placement Analysis & Skill Gap Recommendation System
### Final Year B.Tech CSE Project Walkthrough

This notebook demonstrates the end-to-end Machine Learning pipeline:
1. Data Loading & Exploratory Data Analysis (EDA)
2. Feature Preprocessing & Scaling (`StandardScaler`)
3. Model Training & Comparison:
   - Logistic Regression
   - Decision Tree Classifier
   - Random Forest Classifier
4. Real Model Evaluation (Accuracy, Precision, Recall, F1-Score, Confusion Matrix)
5. Job Role Matching & Set-Theoretic Skill Gap Identification
6. Learning Recommendations

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

print('All libraries successfully imported!')

In [ ]:
# Load synthetic benchmark student dataset
data_path = os.path.join('..', 'data', 'students.csv')
df = pd.read_csv(data_path)
print(f'Dataset Shape: {df.shape}')
df.head()

In [ ]:
# Check Class Distribution and Missing Values
print('Missing Values:\n', df.isnull().sum())
print('\nPlacement Status Distribution:')
print(df['Placement_Status'].value_counts(normalize=True))

In [ ]:
# Feature Separation and Train-Test Split
feature_cols = [
    'CGPA', 'Tenth_Percentage', 'Twelfth_Percentage', 'Backlogs',
    'Python', 'Cpp', 'Java', 'SQL', 'Excel', 'HTML_CSS', 'JavaScript',
    'ML_Knowledge', 'Data_Analysis', 'Projects', 'Internships',
    'Certifications', 'Aptitude_Score', 'Communication_Score'
]
X = df[feature_cols]
y = df['Placement_Status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Standardize numerical continuous features
num_cols = ['CGPA', 'Tenth_Percentage', 'Twelfth_Percentage', 'Aptitude_Score', 'Communication_Score']
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

print(f'Training Samples: {len(X_train)}, Testing Samples: {len(X_test)}')

In [ ]:
# Train and Evaluate Candidate Models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=7, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1-Score': f1_score(y_test, y_pred, zero_division=0)
    }

results_df = pd.DataFrame(results).T
results_df

In [ ]:
# Plot Model Comparison
plt.figure(figsize=(8, 4))
results_df[['Accuracy', 'F1-Score']].plot(kind='bar', figsize=(8, 4), colormap='viridis')
plt.title('Model Performance Comparison on Test Set')
plt.ylabel('Score')
plt.ylim(0, 1.1)
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Job Role Skill Matching Demonstration
job_roles = pd.read_csv(os.path.join('..', 'data', 'job_roles.csv'))

sample_student_skills = {'Python', 'Excel', 'Data Analysis'}
target_role = job_roles[job_roles['Role_Name'] == 'Data Analyst'].iloc[0]
required_skills = set([s.strip() for s in target_role['Required_Skills'].split(',')])

matching = sample_student_skills.intersection(required_skills)
gap = required_skills - sample_student_skills
match_pct = (len(matching) / len(required_skills)) * 100

print(f"Target Role: {target_role['Role_Name']}")
print(f"Required Skills: {required_skills}")
print(f"Matching Skills: {matching}")
print(f"Skill Gap (Missing): {gap}")
print(f"Match Percentage: {match_pct:.1f}%")